In [3]:
import pandas as pd
import os
from sklearn.metrics import f1_score

path = 'data\\hwu'

df_train = pd.read_csv(os.path.join(path, 'train.csv'), sep=',', header=0, names=['text', 'intent'])
df_val = pd.read_csv(os.path.join(path,'val.csv'), sep=',', header=0, names=['text', 'intent'])
df_test = pd.read_csv(os.path.join(path,'test.csv'), sep=',', header=0, names=['text', 'intent'])

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

intents = pd.concat([df_train['intent'], df_val['intent'], df_test['intent']])

encoder.fit(intents)

df_train['label'] = encoder.transform(df_train['intent'])
df_val['label'] = encoder.transform(df_val['intent'])
df_test['label'] = encoder.transform(df_test['intent'])

df_train

Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,intent,label
0,what alarms do i have set right now,alarm_query,0
1,checkout today alarm of meeting,alarm_query,0
2,report alarm settings,alarm_query,0
3,see see for me the alarms that you have set to...,alarm_query,0
4,is there an alarm for ten am,alarm_query,0
...,...,...,...
8949,how hot is it in miami,weather_query,63
8950,will it snow next week,weather_query,63
8951,am i gonna need rain boots,weather_query,63
8952,should i bring warm clothes,weather_query,63


In [5]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# 1. Huấn luyện mô hình Word2Vec trên dữ liệu text của bạn
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)
# 2. Viết hàm để chuyển mỗi câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    words = text.split()
    vectors = []
    for w in words:
        if w in model.wv:
            vectors.append(model.wv[w])
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    avg_vector = np.mean(vectors, axis=0)
    return avg_vector

# 3. Tạo dữ liệu train/val/test X_train_avg, X_val_avg, X_test_avg
x_train_avg = np.vstack(df_train['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))
x_val_avg = np.vstack(df_val['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))
x_test_avg = np.vstack(df_test['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))


# 4. Xây dựng mô hình Sequential của Keras
num_classes = df_train['label'].nunique()

model = Sequential([
Dense(128, activation='relu', input_shape=(w2v_model.vector_size,)),
Dropout(0.5),
Dense(num_classes, activation='softmax')
])

# 5. Compile, huấn luyện và đánh giá mô hình
#Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

#Huấn luyện model
history = model.fit(
    x_train_avg, y_train,
    validation_data=(x_val_avg, y_val),
    epochs=100,
    batch_size=32,
    verbose=1
)

#Đánh giá model
test_loss_w2v, test_acc = model.evaluate(x_test_avg, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_pred_avg = np.argmax(model.predict(x_test_avg), axis=1)

# Macro F1
f1_avg = f1_score(y_test, y_pred_avg, average='macro')

Epoch 1/100


d:\College\year6s1\nlp-dl\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0195 - loss: 4.1433 - val_accuracy: 0.0390 - val_loss: 4.1117
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0342 - loss: 4.1027 - val_accuracy: 0.0520 - val_loss: 4.0589
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0427 - loss: 4.0359 - val_accuracy: 0.0493 - val_loss: 3.9678
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0585 - loss: 3.9375 - val_accuracy: 0.0799 - val_loss: 3.8483
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0694 - loss: 3.8371 - val_accuracy: 0.0929 - val_loss: 3.7353
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0766 - loss: 3.7528 - val_accuracy: 0.1134 - val_loss: 3.6551
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0829 - loss: 3.6780 - val_accuracy: 0.1199 - val_loss: 3.5795
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0925 - loss: 3.6140 - val_accuracy: 0.1533